In [ ]:
from pprint import pprint
from collections import Counter, defaultdict
import re
import json

from matplotlib import pyplot
import pandas as pd

from brickmapper import Mapper
from brickmapper.bedes_parser.bedes_parser import BedesParser
from brickmapper.buildingsync.buildingsync_parser import BuildingSyncParser


In [ ]:
"""Get 229P defns"""
with open('brickmapper/229P_parser/schema_definitions.json', 'r') as file:
    data = json.load(file)
    _229p_defns = list(data.values())
    _229p_defns = [
        {
            "name": n.split("/")[-1],
            "tree": d["tree"],
            "definition": d["description"],
        }
        for n, d in data.items()
    ]

pprint(_229p_defns)

In [ ]:
"""Get BuildingSync defns"""
buildingsync_parser = BuildingSyncParser()
buildingsync_defns = buildingsync_parser._get_term_definitions()

pprint([b for b in buildingsync_defns if b["name"] == "Building"])

In [ ]:
"""Create Mapper"""
mapper = Mapper(
    first_definitions=[{"name": d["name"], "definition": d["definition"]} for d in buildingsync_defns],
    first_index_file="./indices/names_and_definitions/buidingsync.ollama.index",
    second_definitions=[{"name": d["name"], "definition": d["definition"]} for d in _229p_defns],
    second_index_file="./indices/names_and_definitions/229P.ollama.index",
)

In [ ]:
"""Get Mappings"""
top_k = 3
threshold = .3
results = mapper.get_mappings_with_collisions(top_k, threshold)

pd.DataFrame(
    [[name, *r] for name, r in results.items()], 
    columns=["BuildingSync Term", "#1 Best Match", "#2 Best Match", "#3 Best Match"]
).to_excel("229-names-and-defintions-results.xlsx")

pprint(results)

In [ ]:
"""Some quick analysis"""

number_of_suggestions_above_threshold = [len(r) for r in results.values()]
print(f"the threshold was a distance of {threshold}:")
for num_results, num_with_num_results in Counter(number_of_suggestions_above_threshold).items():
    print(f"\t {num_with_num_results} defintitons had {num_results} results below threshold")

print("\n\nhere are some of the defintions with no matches:")
pprint([name for name, r in results.items() if len(r) == 0][:5])

scores_by_rank = defaultdict(list)
for r in results.values():
    for rank, (_, score) in enumerate(r):
        scores_by_rank[rank].append(score)

print("\n\nhere's a histogram of score, groups by the rank of the match")
for rank in range(top_k):
    pyplot.hist(scores_by_rank[rank], alpha=0.5, label=rank+1)
    
pyplot.legend(loc='upper right')
pyplot.show()



In [ ]:
for rank in range(top_k):
    results_with_rank = [r for r in results.items() if len(r[1]) > rank]
    best_matches = sorted(results_with_rank, key=lambda x: 1 * x[1][rank][1])
    print(f"Some of the best matches for rank {rank + 1} are:")
    pprint([(m[0], m[1][rank]) for m in best_matches[:5]])
    print("\n")


In [ ]:
unmatched_terms = [term for term, matches in results.items() if len(matches) == 0]

pprint(unmatched_terms)

In [ ]:
set_of_broken_unmatched_terms = set()
for t in unmatched_terms:
    broken_term = re.findall('[A-Z][^A-Z]*', t)
    set_of_broken_unmatched_terms.update(broken_term)

pprint(set_of_broken_unmatched_terms)

In [ ]:
seen_broken_unmatched_terms = {t: results[t] for t in set_of_broken_unmatched_terms if t in results}

pprint(f"We have seen {len(seen_broken_unmatched_terms)} of the {len(set_of_broken_unmatched_terms)} broken unmatches terms in the original defintions")
pprint(seen_broken_unmatched_terms)

In [ ]:
unseen_broken_unmatched_terms = [t for t in set_of_broken_unmatched_terms if t not in results]

for term in unseen_broken_unmatched_terms[:10]:
    print(term)
    pprint(mapper.get_mappings_for_single_definition({"name": term, 'term_definition': ''}))
    

In [ ]:
mapper.get_mappings_for_single_definition({
    'definition': 'A building is a single structure wholly or partially enclosed '
                'within exterior walls, or within exterior and abutment walls '
                '(party walls), and a roof, affording shelter to persons, '
                'animals, or property. A building can be two or more units '
                'held in the condominium form of ownership that are governed '
                'by the same board of managers.',
  'name': 'Building',}, top_k=3, threshold=0.8)